# Project 1
## 1. Incremental ingestion

In [1]:
#imports 
import os
import json
from datetime import datetime
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("Project1_Group_H")
    .enableHiveSupport()
    .config("spark.sql.autoBroadcastJoinThreshold", "10MB")
    .getOrCreate()
)

In [3]:
# ---- Paths ----
base_path = "/home/jovyan/work"

inbox_path = os.path.join(base_path, "data/inbox")
manifest_path = os.path.join(base_path, "state/manifest.json")
outbox_path = os.path.join(base_path, "data/outbox")
output_path = os.path.join(outbox_path, "trips_enriched.parquet")

# ---- Create required directories ----
os.makedirs(f"{base_path}/state", exist_ok=True)
os.makedirs(outbox_path, exist_ok=True)

# ---- Load manifest if exists ----
if os.path.exists(manifest_path):
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
else:
    manifest = {
    "trip_files": [],
    "lookup_loaded": False
    }


In [4]:
# ---- Load TAXI ZONE LOOKUP once ----
lookup_path = os.path.join(inbox_path, "taxi_zone_lookup.parquet")

if not manifest["lookup_loaded"] and os.path.exists(lookup_path):
    zones_raw = spark.read.parquet(lookup_path)
    zones_raw.cache()
    print("Lookup rows:", zones_raw.count())
    manifest["lookup_loaded"] = True
else:
    print("Lookup already loaded previously.")

# ---- Get all TAXI TRIP files ----
all_files = [f for f in os.listdir(inbox_path)if f.endswith(".parquet") and f != "taxi_zone_lookup.parquet"]

# ---- Select only new taxi trip files (checks file size and name) ----
processed_index = {
    f["filename"]: f["file_size"]
    for f in manifest["trip_files"]
}

new_files = []

for file in all_files:
    full_path = os.path.join(inbox_path, file)
    current_size = os.path.getsize(full_path)

    if file not in processed_index or processed_index[file] != current_size:
        new_files.append(file)

print("New files to process:", new_files)

Lookup already loaded previously.
New files to process: []


In [5]:
# ---- Read new files ----
if not new_files:
    print("No new files found.")
else:
    full_paths = [os.path.join(inbox_path, f) for f in new_files]
    df_raw = spark.read.parquet(*full_paths)
    df_raw = (df_raw.withColumn("source_file", F.input_file_name())
            .withColumn("ingested_at", F.current_timestamp()))
    total_rows = df_raw.count()
    print("Total rows read:", total_rows)

    # metadata for manifest
    for file in new_files:
        file_size = os.path.getsize(os.path.join(inbox_path, file))

        manifest["trip_files"].append({
            "filename": file,
            "file_size": file_size,
            "processed_at": datetime.now().isoformat()
        })

# ---- Save manifest modification ---- 
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)

print("Manifest updated.")

No new files found.
Manifest updated.


## 2. Transformations
### Parse and cast types

In [6]:
df_casted = df_raw.select(
    F.col("VendorID").cast("int"),
    # timestamps
    F.col("tpep_pickup_datetime").cast("timestamp").alias("pickup_datetime"),
    F.col("tpep_dropoff_datetime").cast("timestamp").alias("dropoff_datetime"),

    # ints
    F.col("PULocationID").cast("int"),
    F.col("DOLocationID").cast("int"),
    F.col("passenger_count").cast("int"),

    # floats
    F.col("trip_distance").cast("double"),
    F.col("fare_amount").cast("double"),
    F.col("total_amount").cast("double"),

    #metadata    
    F.col("source_file"),
    F.col("ingested_at")
)

df_casted.printSchema()
df_casted.show(3, truncate=False)

root
 |-- VendorID: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- source_file: string (nullable = false)
 |-- ingested_at: timestamp (nullable = false)

+--------+-------------------+-------------------+------------+------------+---------------+-------------+-----------+------------+-------------------------------------------------------------------+--------------------------+
|VendorID|pickup_datetime    |dropoff_datetime   |PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|total_amount|source_file                                                        |ingested_at               |
+--------+-------------------+-------

In [7]:
zones_raw.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


### Clean
Rules which we discussed during meetup.

| # | Rule | Column(s) | Condition kept |
|---|------|-----------|----------------|
| 1 | Non-null timestamps | `pickup_datetime`, `dropoff_datetime` | `isNotNull()` |
| 2 | Chronological trip | both timestamps | `dropoff > pickup` |
| 3 | Positive distance | `trip_distance` | `> 0` |
| 4 | Valid passenger count | `passenger_count` | `> 0` |
| 5 | Positive fare | `total_amount`, `fare_amount` | `> 0` |
| 6 | Valid location IDs | `PULocationID`, `DOLocationID` | `isNotNull()` |

In [8]:
df_cleaned = df_casted.filter(
    F.col("pickup_datetime").isNotNull() &
    F.col("dropoff_datetime").isNotNull() &
    (F.col("dropoff_datetime") > F.col("pickup_datetime")) &
    (F.col("trip_distance") > 0) &
    (F.col("passenger_count") > 0) &
    ((F.col("total_amount") > 0) & (F.col("fare_amount") > 0)) &
    F.col("PULocationID").isNotNull() &
    F.col("DOLocationID").isNotNull()
)

raw_count = df_casted.count()
after_clean_count = df_cleaned.count()

print(f"Rows input: {raw_count}")
print(f"Rows after clean: {after_clean_count}(removed {raw_count - after_clean_count})")

Rows input: 7052769
Rows after clean: 5472446(removed 1580323)


In [9]:
print("trip_distance <= 0:")
df_casted.filter(
    F.col("trip_distance") <= 0
).select(
    "pickup_datetime", "dropoff_datetime", "trip_distance", "passenger_count", "total_amount"
).show(2, truncate=False)

print("passenger_count < 1:")
df_casted.filter(
    F.col("passenger_count") < 1
).select(
    "pickup_datetime", "dropoff_datetime", "trip_distance", "passenger_count", "total_amount"
).show(2, truncate=False)

print("dropoff_datetime <= pickup_datetime:")
df_casted.filter(
    F.col("pickup_datetime").isNotNull() &
    (F.col("dropoff_datetime") <= F.col("pickup_datetime"))
).select(
    "pickup_datetime", "dropoff_datetime", "trip_distance", "passenger_count"
).show(1, truncate=False)
df_casted.filter(
    F.col("pickup_datetime").isNotNull() &
    (F.col("dropoff_datetime") < F.col("pickup_datetime"))
).select(
    "pickup_datetime", "dropoff_datetime", "trip_distance", "passenger_count"
).show(1, truncate=False)

trip_distance <= 0:
+-------------------+-------------------+-------------+---------------+------------+
|pickup_datetime    |dropoff_datetime   |trip_distance|passenger_count|total_amount|
+-------------------+-------------------+-------------+---------------+------------+
|2025-01-01 00:49:48|2025-01-01 00:49:48|0.0          |1              |20.06       |
|2025-01-01 00:37:43|2025-01-01 00:37:53|0.0          |1              |17.5        |
+-------------------+-------------------+-------------+---------------+------------+
only showing top 2 rows
passenger_count < 1:
+-------------------+-------------------+-------------+---------------+------------+
|pickup_datetime    |dropoff_datetime   |trip_distance|passenger_count|total_amount|
+-------------------+-------------------+-------------+---------------+------------+
|2025-01-01 00:14:47|2025-01-01 00:16:15|0.4          |0              |11.75       |
|2025-01-01 00:39:27|2025-01-01 00:51:51|1.6          |0              |19.1        |


### Deduplication
**Key:** `(pickup_datetime, dropoff_datetime, PULocationID, DOLocationID, trip_distance, vendorID)`  
Two real trips cannot share the same origin, destination, exact times, and distance.

In [10]:
deduplication_key = [
    "pickup_datetime",
    "dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "VendorID",
]

df_dedup = df_cleaned.dropDuplicates(deduplication_key)

after_dedup_count = df_dedup.count()
print(f"Rows after clean: {after_clean_count}")
print(f"Rows after dedup: {after_dedup_count}(removed {after_clean_count - after_dedup_count} duplicates)")

Rows after clean: 5472446
Rows after dedup: 5472446(removed 0 duplicates)


### Derived Columns


In [11]:
df_derived = df_dedup.selectExpr(
    "*",
    "round((unix_timestamp(dropoff_datetime) - unix_timestamp(pickup_datetime)) / 60.0, 2) AS trip_duration_minutes",
    "to_date(pickup_datetime) AS pickup_date"
)

df_derived.select(
    "pickup_datetime", "dropoff_datetime",
    "trip_distance", "trip_duration_minutes", "pickup_date"
).show(5, truncate=False)

+-------------------+-------------------+-------------+---------------------+-----------+
|pickup_datetime    |dropoff_datetime   |trip_distance|trip_duration_minutes|pickup_date|
+-------------------+-------------------+-------------+---------------------+-----------+
|2025-01-01 00:47:58|2025-01-01 00:59:47|1.05         |11.82                |2025-01-01 |
|2025-01-01 00:38:17|2025-01-01 00:56:17|2.2          |18.00                |2025-01-01 |
|2025-01-01 00:12:31|2025-01-01 00:31:32|4.0          |19.02                |2025-01-01 |
|2025-01-01 00:54:53|2025-01-01 01:14:39|3.43         |19.77                |2025-01-01 |
|2025-01-01 00:17:27|2025-01-01 00:39:02|3.92         |21.58                |2025-01-01 |
+-------------------+-------------------+-------------+---------------------+-----------+
only showing top 5 rows


## 3. Zone Enrichment
Cast column names for the zones once, then create separate pickup and dropoff views. the zone table has 265 rows therefore smaller than main data aand we'll use broacast join

In [12]:
zones = zones_raw.select(
    F.col("LocationID").cast("int"),
    F.col("Zone").alias("zone_name"),
    F.col("Borough").alias("borough")
)

pickup_zones = zones.select(
    F.col("LocationID").alias("PULocationID"),
    F.col("zone_name").alias("pickup_zone"),
    F.col("borough").alias("pickup_borough")
)

dropoff_zones = zones.select(
    F.col("LocationID").alias("DOLocationID"),
    F.col("zone_name").alias("dropoff_zone"),
    F.col("borough").alias("dropoff_borough")
)

print("autoBroadcastJoinThreshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

df_enriched = (
    df_derived
    .join(F.broadcast(pickup_zones), on="PULocationID", how="left")
    .join(F.broadcast(dropoff_zones), on="DOLocationID", how="left")
)

df_enriched.select(
    "pickup_datetime", 
    "PULocationID", 
    "pickup_zone", 
    "pickup_borough",
    "DOLocationID", 
    "dropoff_zone", 
    "dropoff_borough"
).show(5, truncate=False)

autoBroadcastJoinThreshold: 10MB
+-------------------+------------+---------------------+--------------+------------+---------------------+---------------+
|pickup_datetime    |PULocationID|pickup_zone          |pickup_borough|DOLocationID|dropoff_zone         |dropoff_borough|
+-------------------+------------+---------------------+--------------+------------+---------------------+---------------+
|2025-01-01 00:47:58|234         |Union Sq             |Manhattan     |162         |Midtown East         |Manhattan      |
|2025-01-01 00:38:17|249         |West Village         |Manhattan     |170         |Murray Hill          |Manhattan      |
|2025-01-01 00:12:31|236         |Upper East Side North|Manhattan     |42          |Central Harlem North |Manhattan      |
|2025-01-01 00:54:53|48          |Clinton East         |Manhattan     |263         |Yorkville West       |Manhattan      |
|2025-01-01 00:17:27|107         |Gramercy             |Manhattan     |239         |Upper West Side South|

### End result

Required output fields must include at least:
* pickup and dropoff timestamps
* pickup and dropoff LocationID
* pickup and dropoff zone name (from lookup)
* passenger_count, trip_distance
* derived: trip_duration_minutes, pickup_date
* metadata: source_file, ingested_at

In [13]:
df_output = df_enriched.select(
    "VendorID",
    "pickup_datetime",
    "dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "pickup_zone",
    "pickup_borough",
    "dropoff_zone",
    "dropoff_borough",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "trip_duration_minutes",
    "pickup_date",
    F.date_format("pickup_date", "yyyy-MM").alias("pickup_month"),
    "source_file",
    "ingested_at"
)

df_output.printSchema()
df_output.show(5, truncate=False)

final_count = df_output.count()
print(f"Input(raw): {raw_count}")
print(f"After cleaning: {after_clean_count}")
print(f"After dedup: {after_dedup_count}")
print(f"Final output: {final_count}")


root
 |-- VendorID: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- pickup_zone: string (nullable = true)
 |-- pickup_borough: string (nullable = true)
 |-- dropoff_zone: string (nullable = true)
 |-- dropoff_borough: string (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- trip_duration_minutes: decimal(24,2) (nullable = true)
 |-- pickup_date: date (nullable = true)
 |-- pickup_month: string (nullable = true)
 |-- source_file: string (nullable = false)
 |-- ingested_at: timestamp (nullable = false)

+--------+-------------------+-------------------+------------+------------+--------------+--------------+--------------+---------------+---------------+-------------

In [14]:
df_output.explain(True)

== Parsed Logical Plan ==
'Project ['VendorID, 'pickup_datetime, 'dropoff_datetime, 'PULocationID, 'DOLocationID, 'pickup_zone, 'pickup_borough, 'dropoff_zone, 'dropoff_borough, 'passenger_count, 'trip_distance, 'fare_amount, 'total_amount, 'trip_duration_minutes, 'pickup_date, 'date_format('pickup_date, yyyy-MM) AS pickup_month#3386, 'source_file, 'ingested_at]
+- Project [DOLocationID#2864, PULocationID#2863, VendorID#2862, pickup_datetime#2860, dropoff_datetime#2861, passenger_count#2865, trip_distance#2866, fare_amount#2867, total_amount#2868, source_file#153, ingested_at#154, trip_duration_minutes#3109, pickup_date#3110, pickup_zone#3141, pickup_borough#3142, dropoff_zone#3144, dropoff_borough#3145]
   +- Join LeftOuter, (DOLocationID#2864 = DOLocationID#3143)
      :- Project [PULocationID#2863, VendorID#2862, pickup_datetime#2860, dropoff_datetime#2861, DOLocationID#2864, passenger_count#2865, trip_distance#2866, fare_amount#2867, total_amount#2868, source_file#153, ingested_at#

In [ ]:
# ---- Append enriched data to outbox parquet ----
(
    df_output
    .repartition("pickup_month")
    .write
    .mode("append")
    .partitionBy("pickup_month")
    .parquet(output_path)
)

print(f"Appended {final_count} rows to {output_path}")


Appended 5472446 rows to /home/jovyan/work/data/outbox/trips_enriched.parquet


In [ ]:
# ---- Incremental monthly borough summary ----
summary_path = os.path.join(outbox_path, "monthly_borough_summary.parquet")

new_months = df_output.select("pickup_month").distinct()

if os.path.exists(summary_path):
    existing_months = spark.read.parquet(summary_path).select("pickup_month").distinct()
    months_to_process = new_months.join(existing_months, on="pickup_month", how="left_anti")
else:
    months_to_process = new_months

if months_to_process.limit(1).count() == 0:
    print("No new months to append in monthly_borough_summary.")
else:
    df_monthly_borough_summary = (
        df_output
        .join(F.broadcast(months_to_process), on="pickup_month", how="inner")
        .groupBy("pickup_month", "pickup_borough")
        .agg(
            F.count("*").alias("trip_count"),
            F.round(F.sum("total_amount"), 2).alias("total_revenue"),
            F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
            F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_minutes")
        )
    )

    (
        df_monthly_borough_summary
        .repartition("pickup_month")
        .write
        .mode("append")
        .partitionBy("pickup_month")
        .parquet(summary_path)
    )

    print(f"Appended monthly borough summary to {summary_path}")
